# 01 Epoch Capabilities Index: model capability table

**Purpose.** This notebook builds `model_capability.csv`, the capability table for Q1 (which models are strong enough to matter) and the base table for Q2 and Q8. It downloads the Epoch Capabilities Index (ECI) file, inspects it, adds our own accessibility and licence columns beside Epoch's labels and saves one clean table.

**Question served.** Q1 (the one question) and the evidence chapters Q2 and Q8.

**How to run.** Run all cells from top to bottom. Nothing needs to be edited. If the download is blocked, the notebook uses the file already in `raw/` or the copy in `manual/` (see Section 3). The last section lists the file to copy by hand into the story's `data/` folder.

## At a glance: links, authorisation, download, timing

| Item | Detail |
|---|---|
| Kind | Download of one CSV |
| Where it comes from | Epoch AI. Data page https://epoch.ai/data . File https://epoch.ai/data/eci_scores.csv . Chart page https://epoch.ai/benchmarks?view=graph&tab=eci |
| Licence and authorisation | Creative Commons Attribution (version not stated on the pages read, project notes say CC BY 4.0). Credit Epoch AI. No account or key needed. Details in Section 1 |
| Download size | about 34 KB (277 models in the file of 2026-10-18) |
| Expected time | download 1 to 5 seconds, whole notebook under 10 seconds. Measured in our test run: 1.7 to 3.3 seconds, using the local copy because the test machine could not reach epoch.ai |
| Data as of | `processed/data_currency.csv` (written by the last cell) |
| To update | Run the notebook again. It downloads a fresh file. See the update section below |

## 1. Source, reference links and authorisation

| Item | Detail |
|---|---|
| Publisher | Epoch AI |
| Dataset | Epoch Capabilities Index (ECI) scores per model |
| Data page | https://epoch.ai/data |
| Benchmarking Hub (parent project) | https://epoch.ai/benchmarks |
| ECI page with the chart "Epoch Capabilities Index over time" | https://epoch.ai/benchmarks?view=graph&tab=eci . Use it to compare our table with what Epoch shows on screen. On 18 Oct 2026 the chart said 260 results while the downloaded file has 274 rows, so the chart applies a filter or is a different snapshot. Do not expect the two counts to match |
| Download page | https://epoch.ai/data (list "Download our data", entry "Capabilities & benchmarking", ZIP, updated 17 Oct 2026 when a team member looked) |
| File used | https://epoch.ai/data/eci_scores.csv |
| Method note | https://epoch.ai/blog/open-models-report (Table A1 defines the accessibility labels, read 18 Oct 2026) |
| Licence | Creative Commons Attribution. The Epoch data page says the data is "free to use, distribute, and reproduce provided the source and authors are credited". The version number is not stated on the page we read, and earlier project notes say CC BY 4.0, so confirm the version on the dataset page before the final submission. The ECI page footer shows the label "CC-BY" without a version number (screenshot 18 Oct 2026). The licence text is at https://creativecommons.org/licenses/by/4.0/ |
| Authorisation | No account, key or login needed. The file is public. We do not scrape any page. |
| Citation | Epoch AI, 'Capabilities & benchmarking'. Published online at epoch.ai. Retrieved from https://epoch.ai/benchmarks [online resource] |
| Snapshots we hold | 2026-10-14: 274 models, 33,807 bytes, downloaded for the Part 2 attention app (`a3_p2_attention_app/data/raw/eci_scores.csv`). 2026-10-18: 277 models, 34,152 bytes, hand download, now the copy in `Data/raw/epoch/` |
| Our own licence column | The `licence_class_ours` column comes from our reading of the licence pages on 18 Oct 2026, listed in Section 2. It is not Epoch's work and must not be credited to Epoch. |

## 2. Table structure

### 2.1 Structure of the source file `eci_scores.csv`

One row per model. 277 rows and 11 columns in the hand download of 2026-10-18 (274 rows on 2026-10-14). All names below are as printed in the source.

| Source column | Type | Meaning | Missing values |
|---|---|---|---|
| `Model` | text | Epoch's model name, unique per row | none |
| `Display name` | text | Name for display, usually equal to `Model` | none |
| `eci` | float | Capability score on Epoch's index (higher is better) | none |
| `eci_ci_low`, `eci_ci_high` | float | Lower and upper bound of the confidence interval | a few rows |
| `date` | date text | Release date | none |
| `Organization` | text | Developer. Several developers are joined by a comma | about 17 rows |
| `Country (of organization)` | text | Country of the developer | about 17 rows |
| `Model accessibility` | text | Epoch's label: API access, Open weights (unrestricted), Open weights (restricted use), Open weights (non-commercial), Hosted access (no API), Unreleased, Unknown | none |
| `Accessibility group` | text | Closed weights or open weights | none |
| `model_versions` | text | Always empty in the snapshot | all rows |

### 2.2 Structure of the output `processed/model_capability.csv`

| Output column | Type | How it is made |
|---|---|---|
| `model` | text | `Model`, the join key to other tables |
| `display_name` | text | `Display name` |
| `eci`, `eci_ci_low`, `eci_ci_high` | float | copied |
| `eci_rank` | int | rank by `eci`, 1 is highest, ties share the lowest rank number |
| `release_date` | date | `date` parsed to a date |
| `organization` | text | `Organization` as given |
| `organization_primary` | text | first developer before the first comma |
| `country` | text | `Country (of organization)` |
| `is_china_based` | bool | `country` equals China |
| `epoch_accessibility` | text | `Model accessibility`, Epoch's own label, never edited |
| `epoch_accessibility_group` | text | `Accessibility group` |
| `access_class_epoch` | text | api_only, open_weights, hosted_or_unreleased or unknown, a simple grouping of Epoch's label |
| `licence_class_ours` | text | mit, apache_2_0, custom_threshold, api_only or not_reviewed. See Section 5 |
| `licence_basis` | text | the page we read and the date, or empty |

## 3. Set up the paths and download helper

The notebook writes the files it downloads to `raw/` inside this folder and the tables it builds to `processed/`. Files you saved by hand live in `manual/` inside this folder. `get_file()` tries a fresh download first, then the earlier download, then the manual copy.

In [1]:
import hashlib
import json
import shutil
import time
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import requests

T0 = time.time()
FOLDER = "eci_scores"
cwd = Path.cwd()
if cwd.name == FOLDER:
    HERE = cwd
elif (cwd / FOLDER).exists():
    HERE = cwd / FOLDER
else:
    raise FileNotFoundError(f"Open this notebook from the folder {FOLDER}. Current folder: {cwd}")

MANUAL = HERE / "manual"                       # hand-downloaded source files (optional)
STORY_DATA = HERE.parents[1] / "data"          # the story's own data folder. The notebook never writes here.
RAW = HERE / "raw"               # files this notebook downloads, kept exactly as received
OUT = HERE / "processed"         # tables this notebook writes
for folder in (RAW, OUT):
    folder.mkdir(parents=True, exist_ok=True)
LOG_PATH = RAW / "retrieval_log.json"
print("Notebook folder:", HERE)
print("Manual folder  :", MANUAL)
print("Story data     :", STORY_DATA, "(copy files here by hand, see the last section)")


HEADERS = {"User-Agent": "UTS-36104-A3-student-project (academic use, no contact data stored)"}
TIMEOUT = 120
REFRESH = False  # False: reuse the file already in raw/ (same snapshot as the A3P1 app). True: try a fresh download first.


def record(source_id: str, url: str, path: Path, how: str) -> None:
    """Add (or replace) the retrieval-log entry for one raw file."""
    content = path.read_bytes()
    log = json.loads(LOG_PATH.read_text(encoding="utf-8")) if LOG_PATH.exists() else {}
    log[source_id] = {
        "url": url,
        "file": path.name,
        "how": how,
        "retrieved_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "bytes": len(content),
        "sha256": hashlib.sha256(content).hexdigest(),
    }
    LOG_PATH.write_text(json.dumps(log, indent=2), encoding="utf-8")
    print(f"logged {source_id}: {len(content):,} bytes via {how}")


def get_file(source_id: str, url: str, target: str, manual_name: str = None) -> Path:
    """Return the path of a raw file. Order: fresh download, earlier download in raw/, manual copy."""
    path = RAW / target
    if REFRESH or not path.exists():
        try:
            response = requests.get(url, headers=HEADERS, timeout=TIMEOUT)
            response.raise_for_status()
            path.write_bytes(response.content)
            record(source_id, url, path, "downloaded")
            return path
        except Exception as err:
            print(f"Download of {source_id} failed: {err}")
    manual = MANUAL / (manual_name or target)
    candidates = [p for p in (path, manual) if p.exists()]
    if candidates:
        best = max(candidates, key=lambda p: p.stat().st_mtime)   # the newest copy wins
        if best == path:
            print(f"Using the earlier download {path.name}")
        else:
            shutil.copy2(best, path)
            record(source_id, url, path, f"copied from manual folder {manual.parent.name}")
        return path
    raise FileNotFoundError(
        f"Could not get {source_id}. Open {url} in a browser, save the file as '{manual_name or target}' "
        f"inside {MANUAL}, then run this cell again.")

Notebook folder: /home/claude/work/A3P2/stories/3_reported_incidents/notebooks/eci_scores
Manual folder  : /home/claude/work/A3P2/stories/3_reported_incidents/notebooks/eci_scores/manual
Story data     : /home/claude/work/A3P2/stories/3_reported_incidents/data (copy files here by hand, see the last section)


## 4. Acquire the file

In [2]:
ECI_URL = "https://epoch.ai/data/eci_scores.csv"
eci_path = get_file("eci_scores", ECI_URL, "eci_scores.csv")
eci_raw = pd.read_csv(eci_path)
print(eci_raw.shape)
eci_raw.head()

Using the earlier download eci_scores.csv
(277, 11)


,Model,Display name,eci,eci_ci_low,eci_ci_high,date,Organization,Country (of organization),Model accessibility,Accessibility group,model_versions
0,Claude Opus 5.5,Claude Opus 5.5,166.82,163.54,170.99,2026-09-22,Anthropic,United States of America,API access,Closed weights,NaN
1,GPT-6 Astra,GPT-6 Astra,166.41,163.35,171.23,2026-09-03,OpenAI,United States of America,API access,Closed weights,NaN
2,GPT-6.1 Sol,GPT-6.1 Sol,166.32,163.17,170.82,2026-09-29,OpenAI,United States of America,API access,Closed weights,NaN
3,Claude Fable 5.1,Claude Fable 5.1,164.65,161.92,168.48,2026-09-01,Anthropic,United States of America,API access,Closed weights,NaN
4,Gemini 4 Argon,Gemini 4 Argon,164.56,161.67,168.70,2026-09-30,Google DeepMind,United States of America,Limited access,Other,NaN


## 5. Inspect

Check the shape, the types, the missing values and the labels before changing anything.

In [3]:
expected = ["Model", "Display name", "eci", "eci_ci_low", "eci_ci_high", "date", "Organization",
            "Country (of organization)", "Model accessibility", "Accessibility group", "model_versions"]
missing_cols = [c for c in expected if c not in eci_raw.columns]
extra_cols = [c for c in eci_raw.columns if c not in expected]
print("Missing columns:", missing_cols)
print("New columns not in our structure table:", extra_cols)
assert not missing_cols, "Epoch changed the file structure. Update Section 2 and the rename map below."

print()
print(eci_raw.dtypes)
print()
print("Missing values per column:")
print(eci_raw.isna().sum())
print()
print("Release dates:", eci_raw["date"].min(), "to", eci_raw["date"].max())
print("Duplicated model names:", eci_raw["Model"].duplicated().sum())
print()
print(eci_raw["Model accessibility"].value_counts())
print()
print(eci_raw["Country (of organization)"].value_counts(dropna=False).head(8))

Missing columns: []
New columns not in our structure table: []

Model                            str
Display name                     str
eci                          float64
eci_ci_low                   float64
eci_ci_high                  float64
date                             str
Organization                     str
Country (of organization)        str
Model accessibility              str
Accessibility group              str
model_versions               float64
dtype: object

Missing values per column:
Model                          0
Display name                   0
eci                            0
eci_ci_low                     2
eci_ci_high                    2
date                           0
Organization                  17
Country (of organization)     17
Model accessibility            0
Accessibility group            0
model_versions               277
dtype: int64

Release dates: 2023-02-24 to 2026-10-07
Duplicated model names: 0

Model accessibility
API access             

## 6. Our licence reading (manual layer)

Epoch's accessibility label and the licence text can disagree. Example: Epoch labels Kimi K3 "Open weights (non-commercial)" but the Kimi K3 License we read on 18 Oct 2026 allows commercial use with a Model-as-a-Service revenue condition and exempts internal use. Q8 therefore classifies by the licence text and shows Epoch's label beside it.

The table below holds only the models whose licence page we opened. Every other open-weight model stays `not_reviewed`. Names must match the `Model` column exactly. The sources are in the Gemini-check folder of this project (`05 Gemini Deep Research/Licence Check for Open-Weight Models`) and each row lists the page.

This is not legal advice. Read the full licence text with counsel before any deployment claim.

In [4]:
LICENCE_READING = {
    # Epoch model name: (licence_class_ours, licence_basis)
    "Kimi K3": ("custom_threshold",
                "Kimi K3 License, https://huggingface.co/moonshotai/Kimi-K3 and github.com/MoonshotAI/Kimi-K3, read 2026-10-18. "
                "Model-as-a-Service revenue over US$20 million in 12 months needs a separate agreement. Internal use exempt."),
    "DeepSeek V4 Pro 0813": ("mit",
                "MIT License on https://huggingface.co/deepseek-ai/DeepSeek-V4-Pro, read 2026-10-18. "
                "The page covers the V4-Pro family. Confirm the 0813 release has the same file."),
    "GLM-5": ("mit",
                "Field 'License: mit' on https://huggingface.co/zai-org/GLM-5, read 2026-10-18. "
                "No further licence document, per a team member's own check of the repository."),
    "MiniMax-M3": ("custom_threshold",
                "MiniMax Community License, https://huggingface.co/MiniMaxAI/MiniMax-M3, read 2026-10-18. "
                "Notice always required, separate authorisation above US$20 million yearly revenue, military use prohibited."),
    "Qwen 3.8 27B": ("apache_2_0",
                "Apache License 2.0 shown for Qwen3.8-27B on https://huggingface.co/Qwen, read 2026-10-18. "
                "Confirm that Epoch's 'Qwen 3.8 27B' is the same release."),
}

def group_access(label: str) -> str:
    """Simple grouping of Epoch's own label. Epoch's label itself is kept untouched in epoch_accessibility."""
    if label == "API access":
        return "api_only"
    if isinstance(label, str) and label.startswith("Open weights"):
        return "open_weights"
    if label in ("Hosted access (no API)", "Unreleased"):
        return "hosted_or_unreleased"
    return "unknown"

print("Models with a manual licence reading:", len(LICENCE_READING))
absent = [m for m in LICENCE_READING if m not in set(eci_raw["Model"])]
print("Names not found in the current file (check spelling or Epoch renames):", absent)

Models with a manual licence reading: 5
Names not found in the current file (check spelling or Epoch renames): []


## 7. Transform

In [5]:
rename = {
    "Model": "model",
    "Display name": "display_name",
    "date": "release_date",
    "Organization": "organization",
    "Country (of organization)": "country",
    "Model accessibility": "epoch_accessibility",
    "Accessibility group": "epoch_accessibility_group",
}
cap = eci_raw.rename(columns=rename).drop(columns=["model_versions"])
cap["release_date"] = pd.to_datetime(cap["release_date"], errors="coerce")
cap["eci_rank"] = cap["eci"].rank(method="min", ascending=False).astype(int)
cap["organization_primary"] = cap["organization"].str.split(",").str[0].str.strip()
cap["is_china_based"] = cap["country"].eq("China")
cap["access_class_epoch"] = cap["epoch_accessibility"].map(group_access)

cap["licence_class_ours"] = cap["model"].map(lambda m: LICENCE_READING.get(m, (None, None))[0])
cap["licence_basis"] = cap["model"].map(lambda m: LICENCE_READING.get(m, (None, None))[1])
# API-only models need no licence reading: the weights are not released.
api_only = cap["licence_class_ours"].isna() & cap["access_class_epoch"].eq("api_only")
cap.loc[api_only, "licence_class_ours"] = "api_only"
cap["licence_class_ours"] = cap["licence_class_ours"].fillna("not_reviewed")

cols = ["model", "display_name", "eci", "eci_ci_low", "eci_ci_high", "eci_rank", "release_date",
        "organization", "organization_primary", "country", "is_china_based",
        "epoch_accessibility", "epoch_accessibility_group", "access_class_epoch",
        "licence_class_ours", "licence_basis"]
cap = cap[cols].sort_values("eci", ascending=False).reset_index(drop=True)
cap.head(10)

,model,display_name,eci,eci_ci_low,eci_ci_high,eci_rank,release_date,organization,organization_primary,country,is_china_based,epoch_accessibility,epoch_accessibility_group,access_class_epoch,licence_class_ours,licence_basis
0,Claude Opus 5.5,Claude Opus 5.5,166.82,163.54,170.99,1,2026-09-22,Anthropic,Anthropic,United States of America,False,API access,Closed weights,api_only,api_only,NaN
1,GPT-6 Astra,GPT-6 Astra,166.41,163.35,171.23,2,2026-09-03,OpenAI,OpenAI,United States of America,False,API access,Closed weights,api_only,api_only,NaN
2,GPT-6.1 Sol,GPT-6.1 Sol,166.32,163.17,170.82,3,2026-09-29,OpenAI,OpenAI,United States of America,False,API access,Closed weights,api_only,api_only,NaN
3,Claude Fable 5.1,Claude Fable 5.1,164.65,161.92,168.48,4,2026-09-01,Anthropic,Anthropic,United States of America,False,API access,Closed weights,api_only,api_only,NaN
4,Gemini 4 Argon,Gemini 4 Argon,164.56,161.67,168.70,5,2026-09-30,Google DeepMind,Google DeepMind,United States of America,False,Limited access,Other,unknown,not_reviewed,NaN
5,Claude Sonnet 5.5,Claude Sonnet 5.5,164.45,161.35,168.26,6,2026-09-28,Anthropic,Anthropic,United States of America,False,API access,Closed weights,api_only,api_only,NaN
6,Claude Opus 5,Claude Opus 5,162.60,160.00,165.75,7,2026-07-24,Anthropic,Anthropic,United States of America,False,API access,Closed weights,api_only,api_only,NaN
7,GPT-6 Sol,GPT-6 Sol,162.55,160.03,165.84,8,2026-09-22,OpenAI,OpenAI,United States of America,False,API access,Closed weights,api_only,api_only,NaN
8,Claude Fable 5,Claude Fable 5,162.13,159.53,166.32,9,2026-06-09,Anthropic,Anthropic,United States of America,False,API access,Closed weights,api_only,api_only,NaN
9,GPT-5.5 Pro,GPT-5.5 Pro,161.97,158.97,165.09,10,2026-04-23,OpenAI,OpenAI,United States of America,False,API access,Closed weights,api_only,api_only,NaN


## 8. Validate

These checks stop the notebook if the table breaks a rule the dashboard relies on.

In [6]:
problems = []
if cap["model"].duplicated().any():
    problems.append("duplicated model names")
if cap["eci"].isna().any():
    problems.append("missing eci")
if cap["release_date"].isna().any():
    problems.append("unparsed release dates")
ci = cap.dropna(subset=["eci_ci_low", "eci_ci_high"])
if not ((ci["eci_ci_low"] <= ci["eci"]) & (ci["eci"] <= ci["eci_ci_high"])).all():
    problems.append("eci outside its own confidence interval")
if (cap["release_date"] > pd.Timestamp.now() + pd.Timedelta(days=1)).any():
    problems.append("release date in the future")
unknown_labels = set(cap["epoch_accessibility"]) - {
    "API access", "Open weights (unrestricted)", "Open weights (restricted use)",
    "Open weights (non-commercial)", "Hosted access (no API)", "Unreleased", "Unknown"}
if unknown_labels:
    problems.append(f"new Epoch accessibility labels: {unknown_labels}")
print("Problems:", problems if problems else "none")
assert not problems or problems == [f"new Epoch accessibility labels: {unknown_labels}"], problems

print()
print("Rows:", len(cap))
print("Rows by country (top 5):")
print(cap["country"].value_counts(dropna=False).head(5))
print()
print("Our licence class by Epoch label:")
print(pd.crosstab(cap["epoch_accessibility"], cap["licence_class_ours"]))

Problems: ["new Epoch accessibility labels: {'Limited access'}"]

Rows: 277
Rows by country (top 5):
country
United States of America    161
China                        76
France                       17
NaN                          17
United Arab Emirates          4
Name: count, dtype: int64

Our licence class by Epoch label:
licence_class_ours             apache_2_0  api_only  custom_threshold  mit  \
epoch_accessibility                                                          
API access                              0       122                 0    0   
Hosted access (no API)                  0         0                 0    0   
Limited access                          0         0                 0    0   
Open weights (non-commercial)           0         0                 1    0   
Open weights (restricted use)           0         0                 1    0   
Open weights (unrestricted)             1         0                 0    2   
Unknown                                 0     

## 9. Save

In [7]:
cap.to_csv(OUT / "model_capability.csv", index=False, encoding="utf-8")
print("saved", OUT / "model_capability.csv", len(cap), "rows")

saved /home/claude/work/A3P2/stories/3_reported_incidents/notebooks/eci_scores/processed/model_capability.csv 277 rows


## 10. China-based models in the top 30 (quick look)

A sanity view for the team, not a result. It shows how many of the highest-scoring models come from China-based developers and how Epoch labels them.

In [8]:
top = cap.head(30)
print("China-based share of the top 30:", f"{top['is_china_based'].mean():.0%}")
top.loc[top["is_china_based"], ["model", "eci", "release_date", "organization", "epoch_accessibility", "licence_class_ours"]]

China-based share of the top 30: 10%


,model,eci,release_date,organization,epoch_accessibility,licence_class_ours
16,Kimi K3,157.42,2026-07-16,Moonshot,Open weights (non-commercial),custom_threshold
19,Qwen 3.8 Max,156.88,2026-08-02,Alibaba,API access,api_only
29,GLM-5.3,155.91,2026-08-14,Z.ai (Zhipu AI),API access,api_only


## How to update, and what to do if the download is blocked

**Automatic.** Run all cells again. Section 4 downloads https://epoch.ai/data/eci_scores.csv and replaces `raw/eci_scores.csv`. About 5 seconds.

**Manual download (use this if you see `Download of eci_scores failed`).**

1. Open https://epoch.ai/data in a browser. In the list "Download our data" find **Capabilities & benchmarking** (ZIP) and click the download arrow at the right end of the row. The chart download button on the chart pages only gives PNG or SVG pictures and is not the data.
2. Alternatively paste https://epoch.ai/data/eci_scores.csv into the address bar. The browser saves the CSV.
3. If you took the ZIP, unpack it and take `eci_scores.csv` from the folder `epoch_capabilities_index` (that is where it sat in the archive of 2026-10-14. The columns in newer archives were not checked, and Section 5 stops with a clear message if they differ).
4. Save the CSV as `Data\raw\epoch\eci_scores.csv`, replacing the old copy. Keep the name.
5. Run the notebook again. If the download still fails, the notebook compares the file in `raw/` with the one in `manual/` and uses the **newer** one.

**Update check.** After any update, open `raw/retrieval_log.json` in this folder. The `how` field says `downloaded` when the fresh file was used and `copied from manual folder` when the hand-saved copy was used. Then open `processed/data_currency.csv` and confirm `extracted_utc` and `data_covers_to` are what you expect.

## 11. Run time and limits

**Time to extract.** The download is one file of about 34 KB. Expect 1 to 5 seconds for the download and under 2 seconds for the rest, so a full run takes **under 10 seconds** on a normal connection. The last cell prints the measured time.

**Limits and known risks.**

1. ECI covers models Epoch evaluated. It is not a list of every model on the market.
2. Epoch's model names differ from LiteLLM names, LMArena names and our Search A provider names. A crosswalk table is a separate step and is the biggest risk for Q1.
3. `licence_class_ours` covers five models only. All other open-weight models are `not_reviewed`.
4. Epoch's accessibility label may rest on an older licence. Say "differs from the licence text we read" and do not say "Epoch is wrong" without checking.
5. Country is the developer's country, not where the model runs.

## 12. References

1. Epoch AI, Data page. https://epoch.ai/data
2. Epoch AI, Benchmarking Hub. https://epoch.ai/benchmarks
2b. Epoch AI, ECI chart page. https://epoch.ai/benchmarks?view=graph&tab=eci
3. Epoch AI, ECI scores file. https://epoch.ai/data/eci_scores.csv
4. Epoch AI, Open models report, Table A1 (accessibility definitions). https://epoch.ai/blog/open-models-report
5. Creative Commons, Attribution 4.0 International. https://creativecommons.org/licenses/by/4.0/
6. Licence pages for the five models: https://huggingface.co/moonshotai/Kimi-K3 , https://huggingface.co/deepseek-ai/DeepSeek-V4-Pro , https://huggingface.co/zai-org/GLM-5 , https://huggingface.co/MiniMaxAI/MiniMax-M3 , https://huggingface.co/Qwen
7. Project note: `05 Gemini Deep Research/Licence Check for Open-Weight Models (English and 繁體中文).md`

## Data currency: how up to date is this data?

Every notebook writes `processed/data_currency.csv` with the day it was extracted, the dates the data itself covers, and what we know about when the source started and how often it changes. Use these values for the "data as of" label in the app.

In [9]:
def write_currency(dataset, covers_from, covers_to, snapshot_note, origin_note, frequency):
    """Record how current the data is. extracted_utc is the moment this notebook ran."""
    row = {
        "dataset": dataset,
        "extracted_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "data_covers_from": str(covers_from)[:10] if covers_from is not None else "not stated in the data",
        "data_covers_to": str(covers_to)[:10] if covers_to is not None else "not stated in the data",
        "snapshot_note": snapshot_note,
        "source_origin_note": origin_note,
        "source_update_frequency": frequency,
    }
    pd.DataFrame([row]).to_csv(OUT / "data_currency.csv", index=False, encoding="utf-8")
    for key, value in row.items():
        print(f"{key:24s} {value}")


log = json.loads(LOG_PATH.read_text(encoding="utf-8"))["eci_scores"]
snap = ("file downloaded in this run" if log["how"] == "downloaded"
        else "file copied from Data/raw/epoch, a hand-downloaded copy. Its download day is the file date in File Explorer (the first copy was the Part 2 download of 2026-10-14, replaced by a newer hand download on 2026-10-18)")
write_currency("Epoch Capabilities Index (model_capability.csv)",
               cap["release_date"].min(), cap["release_date"].max(), snap,
               "Epoch publishes ECI inside its Benchmarking Hub. Earliest model in the file is dated by release_date. Epoch's own start date for ECI was not recorded in our notes.",
               "Irregular, as new models are evaluated (typically weekly per earlier project notes)")

dataset                  Epoch Capabilities Index (model_capability.csv)
extracted_utc            2026-11-08T21:52:29Z
data_covers_from         2023-02-24
data_covers_to           2026-10-07
snapshot_note            file downloaded in this run
source_origin_note       Epoch publishes ECI inside its Benchmarking Hub. Earliest model in the file is dated by release_date. Epoch's own start date for ECI was not recorded in our notes.
source_update_frequency  Irregular, as new models are evaluated (typically weekly per earlier project notes)


In [10]:
elapsed = time.time() - T0
print(f"Elapsed time of this run: {elapsed:.1f} seconds")
print("Files written:")
for p in sorted(OUT.glob("*")):
    print(f"  {p.relative_to(HERE)}  ({p.stat().st_size:,} bytes)")

Elapsed time of this run: 0.2 seconds
Files written:
  processed/data_currency.csv  (480 bytes)
  processed/model_capability.csv  (46,475 bytes)


## 12. Copy the table into the story data folder (by hand)

The notebook does not write into the story's `data/` folder. Its outputs stay in `raw/` (files as downloaded) and `processed/` (tables it built). After a run, copy the file listed below from `processed/` into `../../data/` by hand. The page reads only that copy.

The cell below only prints. It shows which file to copy, its number of rows and whether the copy already in `data/` has the same content (line endings are ignored).

In [11]:
TO_COPY = ["model_capability.csv"]   # data_currency.csv stays here as a record of the run
for name in TO_COPY:
    made = OUT / name
    rows = len(pd.read_csv(made))
    kept = STORY_DATA / name
    if not kept.exists():
        state = "not in the story data folder yet. Copy it."
    elif pd.read_csv(kept, keep_default_na=False).equals(pd.read_csv(made, keep_default_na=False)):
        state = "the copy in the story data folder has the same content. Nothing to do."
    else:
        state = "the copy in the story data folder is DIFFERENT. Copy the new file over it."
    print(f"{name}: {rows} rows. From {made.relative_to(HERE)} to {kept}. Status: {state}")


model_capability.csv: 277 rows. From processed/model_capability.csv to /home/claude/work/A3P2/stories/3_reported_incidents/data/model_capability.csv. Status: not in the story data folder yet. Copy it.
